In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import  train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings

warnings.filterwarnings("ignore")


In [2]:
# Step 1: Create an imbalance binary classification
X,y = make_classification(n_samples=1000,n_features=10,n_informative=2,n_redundant=8,weights=[0.9,0.1],flip_y=0,random_state=42)

np.unique(y,return_counts=True)


(array([0, 1]), array([900, 100]))

In [3]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42,stratify=y)

### Experiement 1: Train Logistic Regression Classifier

In [4]:
from sklearn.metrics import classification_report
log_reg = LogisticRegression(C=1,solver='liblinear')
log_reg.fit(X_train,y_train)
y_pred = log_reg.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.95      0.96      0.95       270
           1       0.60      0.50      0.55        30

    accuracy                           0.92       300
   macro avg       0.77      0.73      0.75       300
weighted avg       0.91      0.92      0.91       300



## Experiment 2: Train Random Forest Classifier

In [5]:
rcf_clf = RandomForestClassifier(n_estimators=30,max_depth=3)
rcf_clf.fit(X_train,y_train)
y_pred_rf = rcf_clf.predict(X_test)
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.96      1.00      0.98       270
           1       0.95      0.67      0.78        30

    accuracy                           0.96       300
   macro avg       0.96      0.83      0.88       300
weighted avg       0.96      0.96      0.96       300



### Experiment 3: Train XGBoost

In [6]:
xgb_clf = XGBClassifier(use_label_encoder=False,eval_metric='logloss')
xgb_clf.fit(X_train,y_train)
y_pred_xgb = xgb_clf.predict(X_test)
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       270
           1       0.96      0.80      0.87        30

    accuracy                           0.98       300
   macro avg       0.97      0.90      0.93       300
weighted avg       0.98      0.98      0.98       300



### Experiment 4: Handle class imbalance using SMOTETomek and then Train XGBoost

In [7]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)
X_train_res,y_train_res = smt.fit_resample(X_train,y_train)
np.unique(y_train_res, return_counts=True)

(array([0, 1]), array([619, 619]))

In [8]:
from sklearn.metrics import classification_report
xgb_clf = XGBClassifier(use_label_encode=False,eval_metric='logloss')
xgb_clf.fit(X_train_res,y_train_res)
y_pred_xgb = xgb_clf.predict(X_test)
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       270
           1       0.81      0.83      0.82        30

    accuracy                           0.96       300
   macro avg       0.89      0.91      0.90       300
weighted avg       0.96      0.96      0.96       300



## Track Experiments Using MLFlow

In [9]:
models = [
    (
        "Logistic Regression",
        LogisticRegression(C=1,solver='liblinear'),
        (X_train,y_train),
        (X_test,y_test)
    ),
    (
        "Random Forest",
        RandomForestClassifier(n_estimators=30,max_depth=3),
        (X_train,y_train),
        (X_test,y_test)
    ),
    (
        "XGBClassifier",
        XGBClassifier(use_label_encoder=False,eval_metric='logloss'),
        (X_train,y_train),
        (X_test,y_test)
    ),
    (
        "XGBClassifier with SMOTETomek",
        XGBClassifier(use_label_encoder=False,eval_metric='logloss'),
        (X_train_res,y_train_res),
        (X_test,y_test)
    )
]

In [10]:
reports =[]
for model_name,model,train_set,test_set in models:
    X_train = train_set[0]
    Y_train = train_set[1]
    X_test = test_set[0]
    Y_test = test_set[1]

    model.fit(X_train,Y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test,y_pred,output_dict=True)
    reports.append(report)
    


In [11]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

In [15]:
# Initialize MLflow
mlflow.set_experiment("Anomaly Detection")
mlflow.set_tracking_uri("http://localhost:5000")

for i, element in enumerate(models):
    model_name = element[0]
    model = element[1]
    report = reports[i]

    with mlflow.start_run(run_name=model_name):
            mlflow.log_param("model",model_name)
            mlflow.log_metric('accuracy',report['accuracy'])
            mlflow.log_metric('recall_class_1', report['1']['recall'])
            mlflow.log_metric('recall_class_0',report['0']['recall'])
            mlflow.log_metric('f1_score_macro',report['macro avg']['f1-score'])

            if "XGB" in model_name:
                  mlflow.xgboost.log_model(model,"model")
            else:
                  mlflow.sklearn.log_model(model,"model")

2025/01/31 21:28:08 INFO mlflow.tracking.fluent: Experiment with name 'Anomaly Detection' does not exist. Creating a new experiment.
2025/01/31 21:28:25 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/01/31 21:28:25 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run Logistic Regression at: http://localhost:5000/#/experiments/651788223165844084/runs/249024872166433aa705bf017b65dbeb
🧪 View experiment at: http://localhost:5000/#/experiments/651788223165844084


2025/01/31 21:28:37 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/01/31 21:28:37 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run Random Forest at: http://localhost:5000/#/experiments/651788223165844084/runs/902057d26b0042a08df401db62cb7549
🧪 View experiment at: http://localhost:5000/#/experiments/651788223165844084


2025/01/31 21:28:48 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\ACER\AppData\Local\Temp\tmpjbxaxlr_\model, flavor: xgboost). Fall back to return ['xgboost==2.1.3']. Set logging level to DEBUG to see the full traceback. 
2025/01/31 21:28:48 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/01/31 21:28:48 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run XGBClassifier at: http://localhost:5000/#/experiments/651788223165844084/runs/7e1e5074e23b4538b28df4dcf7fcb15a
🧪 View experiment at: http://localhost:5000/#/experiments/651788223165844084


2025/01/31 21:28:59 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\ACER\AppData\Local\Temp\tmpdhbg00d0\model, flavor: xgboost). Fall back to return ['xgboost==2.1.3']. Set logging level to DEBUG to see the full traceback. 
2025/01/31 21:28:59 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/01/31 21:28:59 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run XGBClassifier with SMOTETomek at: http://localhost:5000/#/experiments/651788223165844084/runs/bacd569a2d50413982c484a4c9c59283
🧪 View experiment at: http://localhost:5000/#/experiments/651788223165844084


## Register the Model

In [37]:
model_name = "RandomForest"
run_id  = input("please type RunID")
model_uri = f"runs:/{run_id}/model"

with mlflow.start_run(run_id=run_id):
    mlflow.register_model(model_uri=model_uri,name=model_name)
    


Successfully registered model 'RandomForest'.
2025/01/31 21:57:17 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: RandomForest, version 1
Created version '1' of model 'RandomForest'.


🏃 View run Random Forest at: http://localhost:5000/#/experiments/651788223165844084/runs/902057d26b0042a08df401db62cb7549
🧪 View experiment at: http://localhost:5000/#/experiments/651788223165844084


In [38]:
import mlflow
model_version=1
model_uri = f"models:/{model_name}/{model_version}"
load_model =mlflow.pyfunc.load_model(model_uri)
y_pred = load_model.predict(X_test)
y_pred[:5]

array([0, 0, 0, 0, 0])

In [39]:
dev_model_uri = f"models:/{model_name}/{model_version}"
prod_model = "anaomaly_detection_prod"
client = mlflow.MlflowClient()
client.copy_model_version(src_model_uri=dev_model_uri,dst_name=prod_model) 

Successfully registered model 'anaomaly_detection_prod'.
Copied version '1' of model 'RandomForest' to version '1' of model 'anaomaly_detection_prod'.


<ModelVersion: aliases=[], creation_timestamp=1738341140840, current_stage='None', description='', last_updated_timestamp=1738341140840, name='anaomaly_detection_prod', run_id='902057d26b0042a08df401db62cb7549', run_link='', source='models:/RandomForest/1', status='READY', status_message=None, tags={}, user_id='', version='1'>